# Laboratório 07: Codificação de Variáveis Categóricas (Encoders)

**Disciplina:** Extração e Preparação de Dados (IBM8915)  
**Professor:** Luís Aramis

---

## 🎯 Objetivo

Aprender a transformar variáveis textuais (strings) em formatos numéricos compatíveis com algoritmos de Machine Learning. Você aprenderá a:

1. **Distinguir** quando usar Encoding Ordinal vs. One-Hot Encoding.
2. **Aplicar** mapeamentos ordinais com `.map()` e dummies com `pd.get_dummies()`.
3. **Evitar** a Armadilha da Variável Dummy usando `drop_first=True`.

---

### 🗺️ Roteiro da Aula

| Parte | Tipo | Tempo estimado |
|---|---|---|
| **Parte 1** | Exemplo Guiado (Descoberta) | ~20 min |
| **Parte 2** | Exercício Prático | ~15 min |
| **Parte 3** | Desafio para Casa (Pipeline Completo) | ~30 min |

> **📌 Pré-requisito:** Este lab pressupõe que você já sabe reduzir a cardinalidade de variáveis categóricas (`lab_06`). Você verá no Desafio por que essa etapa é indispensável antes do Encoding.

In [1]:
# Setup inicial — execute esta célula antes de tudo
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

plt.style.use('ggplot')
import warnings
warnings.filterwarnings('ignore')

print("✅ Bibliotecas carregadas com sucesso!")

✅ Bibliotecas carregadas com sucesso!


---

## Parte 1: Exemplo Guiado — Ordinal vs. One-Hot Encoding

Os modelos de IA só entendem matemática. Precisamos **traduzir** as variáveis textuais.  
Existem duas estratégias principais — e a escolha errada distorce o modelo:

| Estratégia | Quando usar | Ferramenta Pandas |
|---|---|---|
| **Ordinal / Label Encoding** | Há uma **hierarquia** clara (ex: Baixo < Médio < Alto) | `.map(dicionário)` |
| **One-Hot Encoding (Dummies)** | **Sem hierarquia** (ex: TI, RH, Financeiro — nenhum é "maior") | `pd.get_dummies()` |

**⚠️ Armadilha da Variável Dummy:** Ao usar `pd.get_dummies()`, sempre use `drop_first=True` para remover uma coluna redundante e evitar multicolinearidade no modelo.

---

### 🔎 Passo 1 — Pense antes de rodar

O dataset abaixo tem duas colunas categóricas: `Nivel_Acesso` e `Setor`.

> **📝 Antes de executar, responda:**
> - Qual das duas colunas tem hierarquia implícita? Por quê?
> - Quantas colunas numéricas você espera ter no DataFrame final?
>
> _Sua hipótese: [escreva aqui]_

In [2]:
# Criando o dataset de exemplo
df_exemplo = pd.DataFrame({
    'ID_Funcionario': [101, 102, 103, 104],
    'Nivel_Acesso':   ['Baixo', 'Médio', 'Alto', 'Baixo'],  
    'Setor':          ['TI', 'RH', 'TI', 'Financeiro']       
})

print("--- DataFrame Original ---")
print(f"Colunas: {list(df_exemplo.columns)}")
print(f"Shape: {df_exemplo.shape}")
display(df_exemplo)

--- DataFrame Original ---
Colunas: ['ID_Funcionario', 'Nivel_Acesso', 'Setor']
Shape: (4, 3)


,ID_Funcionario,Nivel_Acesso,Setor
0,101,Baixo,TI
1,102,Médio,RH
2,103,Alto,TI
3,104,Baixo,Financeiro


### 🔎 Passo 2 — Ordinal Encoding com `.map()`

Para `Nivel_Acesso`, criamos um dicionário que preserva a hierarquia. Execute e observe o resultado:

In [3]:
# Encoding Ordinal: usamos um dicionário para mapear a hierarquia
mapa_acesso = {'Baixo': 1, 'Médio': 2, 'Alto': 3}
df_exemplo['Nivel_Acesso_Num'] = df_exemplo['Nivel_Acesso'].map(mapa_acesso)

print("Após Ordinal Encoding:")
display(df_exemplo[['ID_Funcionario', 'Nivel_Acesso', 'Nivel_Acesso_Num']])
print("\n💡 Note que a relação matemática 1 < 2 < 3 reflete corretamente Baixo < Médio < Alto.")

Após Ordinal Encoding:


,ID_Funcionario,Nivel_Acesso,Nivel_Acesso_Num
0,101,Baixo,1
1,102,Médio,2
2,103,Alto,3
3,104,Baixo,1



💡 Note que a relação matemática 1 < 2 < 3 reflete corretamente Baixo < Médio < Alto.


### 🔎 Passo 3 — One-Hot Encoding com `pd.get_dummies()`

Para `Setor` (sem hierarquia), criamos colunas binárias separadas. Execute e observe:

> **📝 Você esperava exatamente esse número de colunas? Por que `drop_first=True` removeu a coluna `Setor_Financeiro` e não outra?**
>
> _Sua resposta: [escreva aqui]_

In [ ]:
# One-Hot Encoding: uma coluna binária por categoria
# drop_first=True remove a primeira coluna (ordem alfabética) para evitar multicolinearidade
df_encoded = pd.get_dummies(df_exemplo, columns=['Setor'], drop_first=True, dtype=int)

print("--- DataFrame Transformado para a Máquina ---")
print(f"Shape final: {df_encoded.shape}")
display(df_encoded)

print("\n💡 Com 3 setores (Financeiro, RH, TI), o ________ remove 'Financeiro' (1º em ordem alfabética).")
print("   Quando ______=0 e _______=0, o modelo infere automaticamente que é _______ — sem precisar da coluna!")

--- DataFrame Transformado para a Máquina ---
Shape final: (4, 5)


,ID_Funcionario,Nivel_Acesso,Nivel_Acesso_Num,Setor_RH,Setor_TI
0,101,Baixo,1,0,1
1,102,Médio,2,1,0
2,103,Alto,3,0,1
3,104,Baixo,1,0,0



💡 Com 3 setores (Financeiro, RH, TI), o drop_first=True remove 'Financeiro' (1º em ordem alfabética).
   Quando Setor_RH=0 e Setor_TI=0, o modelo infere automaticamente que é Financeiro — sem precisar da coluna!


### 💬 Reflexão — Parte 1

> 1. **O que aconteceria** se usássemos Ordinal Encoding (`{'TI': 1, 'RH': 2, 'Financeiro': 3}`) no `Setor`?  
>    O modelo aprenderia que `TI < RH < Financeiro`? Isso faz sentido?
>
> 2. **Por que `drop_first=True` existe?** Se tivermos as colunas `Setor_RH` e `Setor_TI`, por que a coluna `Setor_Financeiro` é matematicamente desnecessária?
>
> _Suas respostas: [escreva aqui]_

---

## Parte 2: Exercício Prático — Mão na Massa

Uma loja de carros quer treinar um modelo de precificação. O dataset possui duas colunas categóricas: tamanho do carro (`Tamanho`) e tipo de combustível (`Combustivel`).

---

### 📝 Formule sua hipótese antes de começar

> **Analise as colunas e responda antes de codar:**
> - `Tamanho` tem hierarquia? (`Compacto`, `Sedan`, `SUV`) — justifique.
> - `Combustivel` tem hierarquia? (`Gasolina`, `Diesel`, `Híbrido`, `Elétrico`) — justifique. Cuidado: **há uma armadilha aqui!**
> - Quantas colunas o DataFrame final terá após as duas transformações?
>
> _Minha hipótese: [escreva aqui]_

Execute a célula abaixo para ver os dados e depois resolva o exercício.

In [ ]:
# NÃO ALTERE ESTE CÓDIGO — Geração do cenário do exercício
df_carros = pd.DataFrame({
    'Modelo':      ['Carro A', 'Carro B', 'Carro C', 'Carro D', 'Carro E'],
    'Tamanho':     ['Compacto', 'SUV', 'Sedan', 'Compacto', 'SUV'],
    'Combustivel': ['Gasolina', 'Diesel', 'Hibrido', 'Eletrico', 'Gasolina'],
    'Preco':       [45000, 120000, 75000, 42000, 115000]
})

print(f"Shape original: {df_carros.shape}")
display(df_carros)

In [ ]:
# ✍️ ESCREVA SEU CÓDIGO AQUI

# Passo 1: Aplicar Mapeamento Ordinal na coluna 'Tamanho'
mapa_tamanho = {'Compacto': 1, 'Sedan': 2, 'SUV': 3}
df_carros['Tamanho_Num'] = df_carros['Tamanho'].map(mapa_tamanho)

# Passo 2: Aplicar One-Hot Encoding na coluna 'Combustivel'
df_carros_encoded = pd.get_dummies(df_carros, columns=['Combustivel'], drop_first=True, dtype=int)

# Passo 3: Exiba df_carros_encoded e imprima o shape final
print(f"Shape final: {df_carros_encoded.shape}")
display(df_carros_encoded)

In [ ]:
# ✅ CÉLULA DE VERIFICAÇÃO — Execute após completar o exercício
try:
    assert 'df_carros_encoded' in dir(), "❌ Variável 'df_carros_encoded' não encontrada. Crie-a no Passo 2."
    assert 'Tamanho_Num' in df_carros_encoded.columns or 'Tamanho' in df_carros_encoded.columns, \
        "❌ Coluna de tamanho não encontrada. Crie 'Tamanho_Num' com o mapeamento ordinal."
    assert df_carros_encoded.shape[1] >= 6, (
        f"❌ Shape inesperado: {df_carros_encoded.shape}. "
        "Verifique se o get_dummies foi aplicado em 'Combustivel'."
    )
    assert df_carros_encoded.select_dtypes(include='object').shape[1] == 0 or \
           (set(df_carros_encoded.select_dtypes(include='object').columns) <= {'Modelo', 'Combustivel', 'Tamanho'}), \
        "❌ Ainda há colunas de texto no DataFrame. Verifique as transformações."
    print(f"✅ Ótimo trabalho! Shape final: {df_carros_encoded.shape}")
    print("   O DataFrame está pronto para ser usado em um modelo de ML!")
except AssertionError as e:
    print(e)

### 💬 Reflexão — Parte 2

> 1. **Sua hipótese sobre `Combustivel` estava correta?**  
>    A armadilha: embora `['Gasolina', 'Híbrido', 'Elétrico']` possa sugerir uma escala de "nível de emissões", semanticamente não há consenso sobre a hierarquia de `'Diesel'`. Em análise de dados, **na dúvida, use One-Hot**.
>
> 2. **E se o dataset tivesse 500 tipos de combustíveis diferentes?**  
>    O que teria acontecido com o seu DataFrame ao aplicar `pd.get_dummies()`? O que você faria antes?
>
> _Suas respostas: [escreva aqui]_

---

## Parte 3: Desafio para Casa — Redução + Encoding (Pipeline Completo)

Este é um dos fluxos mais críticos da Engenharia de Dados: **nunca aplique One-Hot Encoding sem antes tratar a cardinalidade**. Caso contrário, o Pandas criará centenas de colunas esparsas e pode estourar a RAM.

---

### ⚠️ Prova do Crime — Pense antes de agir

> **📝 Antes de rodar qualquer célula, estime:**
> - Quantas profissões únicas há no dataset (veja a lista abaixo no código)?
> - Se você aplicar `pd.get_dummies()` **sem** reduzir a cardinalidade, quantas colunas o DataFrame vai ganhar?
> - E **após** agrupar as raras em `'Outros'`, quantas colunas seriam geradas?
>
> _Minha estimativa (sem tratamento): [X] colunas | (com tratamento): [Y] colunas_

**A Missão (Risco de Crédito):**

1. **(Lab 05/Aula 07):** Preencha os nulos da coluna `Idade` com a Mediana.
2. **(Lab 06/Aula 08):** A coluna `Profissao` possui alta cardinalidade. Agrupe todas as profissões com **menos de 3%** de frequência em `'Outros'`.
3. **(Hoje):** Aplique `pd.get_dummies(..., drop_first=True)` **apenas** na coluna `Profissao`.
4. **Comprovação:** Compare o `.shape` do DataFrame antes e depois do tratamento de cardinalidade. Deixe ambos comentados no código.

In [ ]:
# NÃO ALTERE ESTE CÓDIGO — Simulador de Risco de Crédito
np.random.seed(42)
n = 1000

profissoes = np.random.choice(
    ['Professor', 'Engenheiro', 'Vendedor', 'Advogado', 'Padeiro',
     'Youtuber', 'Piloto', 'Mergulhador', 'Astronauta'],
    n,
    p=[0.35, 0.30, 0.25, 0.03, 0.02, 0.02, 0.01, 0.01, 0.01]
)

df_credito = pd.DataFrame({
    'ID_Cliente':   range(1, n + 1),
    'Idade':        np.random.randint(18, 70, n).astype(float),
    'Profissao':    profissoes,
    'Inadimplente': np.random.choice([0, 1], n, p=[0.8, 0.2])
})

# Inserindo valores faltantes em 'Idade'
df_credito.loc[np.random.choice(n, 50, replace=False), 'Idade'] = np.nan

print(f"Shape original: {df_credito.shape}")
print(f"Profissões únicas: {df_credito['Profissao'].nunique()}")
print(f"Nulos em Idade: {df_credito['Idade'].isna().sum()}")
display(df_credito.head(8))

In [ ]:
# 🔎 EXPLORAÇÃO: veja o que aconteceria SEM tratar a cardinalidade
# (rode e compare com o resultado após o tratamento)
df_sem_tratamento = pd.get_dummies(df_credito, columns=['Profissao'], drop_first=True, dtype=int)
print(f"Shape SEM tratamento de cardinalidade: {df_sem_tratamento.shape}")
print(f"Colunas geradas: {list(df_sem_tratamento.columns)}")

In [ ]:
# ✍️ ESCREVA SEU CÓDIGO AQUI

# Trabalhe sobre df_credito — NÃO use df_sem_tratamento

# 1. Preencher nulos de 'Idade' com a mediana
df_credito['Idade'].fillna(df_credito['Idade'].median(), inplace=True)

# 2. Redução de Cardinalidade em 'Profissao' (frequência < 0.03 → 'Outros')
freq_prof = df_credito['Profissao'].value_counts(normalize=True)
profs_raras = freq_prof[freq_prof < 0.03].index
df_credito.loc[df_credito['Profissao'].isin(profs_raras), 'Profissao'] = 'Outros'

# 3. One-Hot Encoding em 'Profissao' (drop_first=True, dtype=int)
df_final = pd.get_dummies(df_credito, columns=['Profissao'], drop_first=True, dtype=int)

# 4. Compare os shapes
print(f"Shape COM tratamento de cardinalidade: {df_final.shape}")
display(df_final.head())

In [ ]:
# ✅ CÉLULA DE VERIFICAÇÃO — Execute após completar o desafio
try:
    assert 'df_final' in dir(), "❌ Variável 'df_final' não encontrada."
    assert df_credito['Idade'].isna().sum() == 0, \
        "❌ Ainda há nulos em 'Idade'. Verifique o fillna."
    assert 'Outros' in df_credito['Profissao'].values or \
           any('Profissao_Outros' in c for c in df_final.columns), \
        "❌ A categoria 'Outros' não foi criada. Verifique o limiar de 3%."
    assert df_final.shape[1] < df_sem_tratamento.shape[1], (
        f"❌ df_final ({df_final.shape[1]} colunas) deveria ter MENOS colunas "
        f"que df_sem_tratamento ({df_sem_tratamento.shape[1]} colunas)."
    )
    ganho = df_sem_tratamento.shape[1] - df_final.shape[1]
    print(f"✅ Pipeline completo! Shape final: {df_final.shape}")
    print(f"   Você economizou {ganho} coluna(s) ao tratar a cardinalidade antes do Encoding!")
except AssertionError as e:
    print(e)

### 💬 Reflexão Final

> 1. **Sua estimativa estava correta?** Quantas colunas o `get_dummies()` gerou sem e com tratamento?
>
> 2. **Escalabilidade:** Imagine um dataset real de crédito com 500.000 clientes e 200 profissões distintas. Se você não tratasse a cardinalidade, a RAM necessária para o `get_dummies()` explodiria. Mas e se algumas profissões raras forem **extremamente relevantes** para prever inadimplência (ex: `'Piloto'` tem renda altíssima)? Como você equilibraria esse trade-off?
>
> 3. **Conexão com o lab_08:** O DataFrame `df_final` agora é 100% numérico. O que isso habilita que era impossível antes?
>
> _Suas respostas: [escreva aqui]_

---

**🎓 Parabéns por concluir o Lab 07!**  
Você agora domina o pipeline `Cardinalidade → Encoding`, um dos mais fundamentais da Engenharia de Features em Machine Learning.